In [ ]:
import matplotlib.pyplot as plt
import torch

from onomatron.data import Vocab, load_names, make_ngram_pairs
from onomatron.mlp import MLP
from onomatron.utils import train

%matplotlib inline

# Load names and build vocab

In [ ]:
names = load_names("../data/processed/names.txt")
vocab = Vocab(names)
print(f"{len(names)} names, vocab of {len(vocab)} chars")

In [ ]:
block_size = 2
pairs = make_ngram_pairs(
    names, vocab, block_size + 1
)  # (N, block_size+1), dernière col = cible
X, Y = pairs[:, :-1], pairs[:, -1]

for i in range(5):
    print(f"{''.join(vocab.itos[int(j)] for j in X[i])} ---> {vocab.itos[int(Y[i])]}")

## Build MLP

Following structure of famous paper [A Neural Probabilistic Language Model]("https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf")

In [ ]:
emb_dim, h_dim = 2, 100
model = MLP(vocab, block_size, emb_dim=emb_dim, h_dim=h_dim, seed=42)

## Find a maximum learning rate

In [ ]:
# Pick lr (full-batch, can take a while)
lrs = (10 ** torch.linspace(-3, 3, 100)).tolist()
# losses = lr_range_test(
#    lambda s: MLP(vocab, block_size, emb_dim, h_dim, seed=s), X, Y, lrs
# )
# best_lr = plot_lr_range(lrs, losses) / 10
best_lr = 0.35  # potentially hardcoded not to redo the full research

## Train the model

In [ ]:
n_steps = 5000
batch_size = 64

model = MLP(vocab, block_size, emb_dim, h_dim, seed=42)
opt = torch.optim.AdamW(model.parameters(), lr=best_lr)  # lr = best_lr du range test
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=best_lr, total_steps=n_steps)
losses = train(
    model, X, Y, opt, steps=n_steps, batch_size=batch_size, scheduler=sched, seed=42
)

plt.plot(losses)
print(f"Final loss achieved is {losses[-1]}")

## Sample from the model

In [ ]:
model.sample(20)